# V21 Pilot (Colab): X-ray-pretrained vs ImageNet-pretrained specialist

**Question:** does chest-X-ray pretraining (TorchXRayVision DenseNet121, trained on ~1M X-rays) beat ImageNet pretraining as init for our benign/malignant specialist?

**How to run:** 1. Runtime > Change runtime type > **T4 GPU**. 2. Runtime > Run all. 3. Paste a **fresh** Kaggle token when asked (regenerate at kaggle.com/settings/api — never reuse one posted in chat). Takes ~40 min.

**Decision rule (write it down before running):** if X-ray-init val beats ImageNet-init val by **>1pp** after 15 epochs each, we fund a full 456px V21 on Kaggle. Else the domain-init idea dies and quota goes to the uncertainty band + demo.

*Known confound, accepted for a pilot: X-ray arm sees 1-channel grayscale (its native input), ImageNet arm sees 3-channel RGB (its native input). This compares usable pipelines, not pure inits.*

In [ ]:
import torch
print('torch:', torch.__version__)
print('cuda:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'Enable T4 GPU: Runtime > Change runtime type'
print('GPU:', torch.cuda.get_device_name(0))
!nvidia-smi -L

In [ ]:
!pip install -q torchxrayvision scikit-learn tqdm
import torchxrayvision as xrv
print('xrv ok')

In [ ]:
import os
from getpass import getpass
os.environ['KAGGLE_API_TOKEN'] = getpass('Paste FRESH Kaggle token (KGAT_...): ')
from kaggle.api.kaggle_api_extended import KaggleApi
api = KaggleApi(); api.authenticate()
print('Kaggle auth OK')
api.dataset_download_files('musakhanalabazy/bone-tumor-dataset3', path='/content/raw', unzip=True)
!ls /content/raw/*/train 2>/dev/null || ls /content/raw

In [ ]:
import hashlib
from pathlib import Path
roots = [p.parent for p in Path('/content/raw').rglob('train/images') if (p.parent / 'labels').is_dir()]
MAIN = roots[0].parent
print('MAIN:', MAIN)
MCLS = {0: 'benign', 1: 'malignant'}

def label_of(img: Path, lbl_dir: Path) -> str:
    lb = lbl_dir / (img.stem + '.txt')
    if not lb.exists():
        lb = lbl_dir / (img.name + '.txt')
        if not lb.exists():
            c = list(lbl_dir.glob(img.stem + '*'))
            lb = c[0] if c else None
    if lb is None or not lb.exists() or lb.stat().st_size == 0:
        return 'normal'
    try:
        return MCLS.get(int(float(open(lb).read().strip().split()[0])), 'malignant')
    except Exception:
        return 'normal'

from PIL import Image

def md5(p: Path) -> str:
    import hashlib as hl
    h = hl.md5()
    with open(p, 'rb') as f:
        for b in iter(lambda: f.read(1 << 20), b''):
            h.update(b)
    return h.hexdigest()

import shutil
D3 = Path('/content/d3')
for split in ['train', 'val', 'test']:
    for c in ['normal', 'benign', 'malignant']:
        (D3 / split / c).mkdir(parents=True, exist_ok=True)
    for img in (MAIN / split / 'images').iterdir():
        if img.suffix.lower() in ('.jpg', '.jpeg', '.png'):
            dst = D3 / split / label_of(img, MAIN / split / 'labels') / img.name
            if not dst.exists():
                try:
                    os.symlink(img, dst)
                except Exception:
                    shutil.copy2(img, dst)
print('staged')
th = {md5(i) for c in ['normal', 'benign', 'malignant'] for i in (D3 / 'test' / c).glob('*')}
drop = 0
for base in [D3 / 'train', D3 / 'val']:
    for img in list(base.rglob('*')):
        if img.is_file() and md5(img) in th:
            img.unlink(); drop += 1
print('dedup dropped:', drop)
SUB = Path('/content/sub')
for split in ['train', 'val']:
    for c in ['benign', 'malignant']:
        (SUB / split / c).mkdir(parents=True, exist_ok=True)
        for img in (D3 / split / c).glob('*'):
            dst = SUB / split / c / img.name
            if not dst.exists():
                try:
                    os.symlink(img, dst)
                except Exception:
                    shutil.copy2(img, dst)
print('train:', sum(1 for _ in (SUB / 'train').rglob('*') if _.is_file()),
      'val:', sum(1 for _ in (SUB / 'val').rglob('*') if _.is_file()))

In [ ]:
import random
import torch.nn as nn
from torch.utils.data import DataLoader
from torch.cuda.amp import GradScaler, autocast
from torchvision import datasets, transforms, models
from sklearn.metrics import accuracy_score
from tqdm import tqdm
device = torch.device('cuda')
IMG, BATCH, EPOCHS, SEED = 224, 32, 15, 21

def aug_tr(gray):
    ts = [transforms.Resize((IMG, IMG))]
    if gray:
        ts.append(transforms.Grayscale(num_output_channels=1))
    ts += [transforms.RandomHorizontalFlip(0.5), transforms.RandomRotation(15),
           transforms.RandomAffine(degrees=10, translate=(0.05, 0.05), scale=(0.9, 1.1)),
           transforms.ColorJitter(brightness=0.15, contrast=0.15),
           transforms.RandAugment(num_ops=2, magnitude=9), transforms.ToTensor()]
    ts.append(transforms.Normalize([0.5], [0.5]) if gray
              else transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]))
    ts.append(transforms.RandomErasing(p=0.25))
    return transforms.Compose(ts)

def aug_ev(gray):
    ts = [transforms.Resize((IMG, IMG))]
    if gray:
        ts.append(transforms.Grayscale(num_output_channels=1))
    ts += [transforms.ToTensor(),
           transforms.Normalize([0.5], [0.5]) if gray
           else transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])]
    return transforms.Compose(ts)

class XRVHead(nn.Module):  # wraps xrv features; bypasses its 18-output head (verified CPU smoke-test)
    def __init__(self, base, ncls=2, drop=0.4):
        super().__init__()
        self.feats = base.features
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(nn.Dropout(drop), nn.Linear(1024, ncls))
    def forward(self, x):
        return self.fc(self.pool(self.feats(x)).flatten(1))

def make_xrv():
    base = xrv.models.DenseNet(weights='densenet121-res224-all')
    return XRVHead(base).to(device)

def make_img():
    m = models.densenet121(weights=models.DenseNet121_Weights.IMAGENET1K_V1)
    m.classifier = nn.Sequential(nn.Dropout(0.4), nn.Linear(m.classifier.in_features, 2))
    return m.to(device)

def run_arm(tag, maker, gray):
    random.seed(SEED); torch.manual_seed(SEED)
    tr = datasets.ImageFolder(str(SUB / 'train'), transform=aug_tr(gray))
    va = datasets.ImageFolder(str(SUB / 'val'), transform=aug_ev(gray))
    tr_ld = DataLoader(tr, batch_size=BATCH, shuffle=True, num_workers=2, pin_memory=True)
    va_ld = DataLoader(va, batch_size=BATCH, shuffle=False, num_workers=2, pin_memory=True)
    model = maker()
    crit = nn.CrossEntropyLoss(label_smoothing=0.05)
    opt = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
    scaler = GradScaler()
    best = 0.0
    for ep in range(1, EPOCHS + 1):
        model.train()
        for x, y in tqdm(tr_ld, desc=f'{tag} {ep}/{EPOCHS}', leave=False):
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            opt.zero_grad()
            with autocast():
                loss = crit(model(x), y)
            scaler.scale(loss).backward()
            scaler.step(opt); scaler.update()
        sched.step()
        P, T = [], []
        model.eval()
        with torch.no_grad():
            for x, y in va_ld:
                P.extend(model(x.to(device)).argmax(1).cpu().tolist())
                T.extend(y.tolist())
        acc = accuracy_score(T, P)
        print(f'{tag} ep{ep}: val={acc:.4f}', flush=True)
        if acc > best:
            best = acc
            torch.save({'model': model.state_dict(), 'acc': acc}, f'/content/{tag}.pt')
    print(f'{tag} BEST VAL: {best:.4f}')
    return best

In [ ]:
xrv_best = run_arm('pilotXRV', make_xrv, gray=True)

In [ ]:
img_best = run_arm('pilotIMG', make_img, gray=False)

In [ ]:
print(f'X-ray-init val: {xrv_best:.4f} | ImageNet-init val: {img_best:.4f} | gap: {xrv_best - img_best:+.4f}')
if xrv_best - img_best > 0.01:
    print('VERDICT: domain init wins -> fund full 456px V21 on Kaggle')
else:
    print('VERDICT: domain init does not transfer -> kill the idea, spend quota on demo + uncertainty band')
from google.colab import files
import glob
best_file = 'pilotXRV.pt' if xrv_best >= img_best else 'pilotIMG.pt'
print('downloading:', best_file, '(copy to local checkpoints/ as pilot_winner.pt)')
files.download('/content/' + best_file)